<a href="https://colab.research.google.com/github/ajith2189/PDF_to_MD_Converter/blob/main/PDF_to_MD_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF to MD Converter

@title
=============================================
CELL 1: Install MinerU in isolated venv
=============================================


In [ ]:
import os

print("Step 1/3: Installing uv (fast Python installer)...")
os.system("pip install uv -q")

print("Step 2/3: Creating isolated venv with Python 3.11...")
os.system("uv venv /content/mineru_env --python 3.11 --seed")

print("Step 3/3: Installing mineru[core] in venv (~5 min)...")
os.system("/content/mineru_env/bin/pip install -U 'mineru[core]' --no-cache-dir")

result = os.popen("/content/mineru_env/bin/mineru --version").read()
print(f"\n✓ MinerU installed: {result.strip()}")
print("✓ Binary at: /content/mineru_env/bin/mineru")



@title
=============================================
CELL 2: Verify GPU + prep model cache
=============================================


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!mkdir -p /content/models_cache
print("\n✓ Ready")



@title
=============================================
CELL 2.5: Download MinerU models (ModelScope) — runs once per session
HuggingFace was refused; ModelScope is the reliable source.
This MUST run before Cell 4, or Cell 4 falls back to HF and fails.
=============================================


In [ ]:
import subprocess
from pathlib import Path

if Path('/root/mineru.json').exists():
    print("✓ Models already downloaded this session")
else:
    print("Downloading pipeline models from ModelScope...")
    r = subprocess.run(
        ['/content/mineru_env/bin/mineru-models-download', '-s', 'modelscope'],
        input='pipeline\n', capture_output=True, text=True
    )
    if 'successfully' in (r.stdout + r.stderr):
        print("✓ Download complete → /root/mineru.json written")
    else:
        print("STDOUT:", r.stdout[-800:])
        print("STDERR:", r.stderr[-800:])



@title
=============================================
CLEANUP: Remove uploaded PDFs and all outputs
Keeps mineru_env venv + model cache (fast re-run)
=============================================


In [ ]:
import shutil
from pathlib import Path

# Folders to clean
targets = [
    '/content/pdfs_input',           # uploaded PDFs
    '/content/markdown_output',       # .md + images + _AI versions
    '/content/final_output',          # clean AI-only folder
    '/content/mineru_batch_input',    # symlinks from batch mode
    '/content/mineru_batch_output',   # mineru temp outputs
    '/content/temp_out_1',            # any per-file temp folders
]

# Also delete generated ZIPs
zip_files = [
    '/content/all_papers_markdown.zip',
    '/content/papers_final.zip',
    '/content/papers_AI_enhanced.zip',
    '/content/papers_AI_only.zip',
    '/content/papers_full.zip',
]

# Delete folders
deleted_folders = 0
for target in targets:
    p = Path(target)
    if p.exists():
        shutil.rmtree(p)
        deleted_folders += 1
        print(f"  ✓ Removed folder: {target}")

# Delete zips
deleted_zips = 0
for zf in zip_files:
    p = Path(zf)
    if p.exists():
        p.unlink()
        deleted_zips += 1
        print(f"  ✓ Removed zip: {zf}")

# Clear any stray temp_out_N folders from old runs
for temp in Path('/content').glob('temp_out_*'):
    if temp.is_dir():
        shutil.rmtree(temp)
        deleted_folders += 1
        print(f"  ✓ Removed folder: {temp}")

# Clear in-memory upload dict (from earlier Cell 3)
try:
    uploaded.clear()
    pdf_paths.clear()
    print("  ✓ Cleared in-memory upload list")
except NameError:
    pass

print(f"\n{'='*50}")
print(f"✓ Cleanup done: {deleted_folders} folders, {deleted_zips} zips removed")
print(f"✓ Kept: mineru venv + model cache (fast re-run)")
print(f"\nReady for new PDF batch — run Cell 3 (upload) to start.")



@title
=============================================
CELL 3: Upload PDFs
=============================================


In [ ]:
from google.colab import files
from pathlib import Path

print("Upload research papers (select multiple)...")
uploaded = files.upload()

input_dir = Path('/content/pdfs_input')
input_dir.mkdir(exist_ok=True)

pdf_paths = []
for orig_name, content in uploaded.items():
    clean_name = orig_name.replace(' ', '_').replace('(', '').replace(')', '')
    dst = input_dir / clean_name
    dst.write_bytes(content)
    pdf_paths.append((orig_name, dst))
    print(f"  ✓ {orig_name} → {clean_name}")

print(f"\n✓ {len(pdf_paths)} PDFs ready")



@title
=============================================
CELL 4: Convert with MinerU — CHUNKED (avoids T4 VRAM OOM)
Root cause of the old "no NVIDIA driver" error: MinerU runs 3 docs
concurrently and OOMs the 15GB T4. Fix = feed it small chunks.
=============================================


In [ ]:
import subprocess
import shutil
from pathlib import Path
import time
import os

output_base = Path('/content/markdown_output')
output_base.mkdir(exist_ok=True)

MINERU = "/content/mineru_env/bin/mineru"



Kill any leftover mineru-api server from earlier experiments.
A frozen server keeps holding GPU VRAM + port 8000 and makes OOM worse.


In [ ]:
subprocess.run(['pkill', '-f', 'mineru-api'], capture_output=True)
time.sleep(3)

# ============ ENV ============
env = os.environ.copy()
env['MINERU_TABLE_ENABLE'] = 'true'
env['MINERU_FORMULA_ENABLE'] = 'false'
env['MINERU_CUSTOM_FORMULA_ENABLE'] = 'false'
env['MINERU_PDF_RENDER_WORKERS'] = '4'
env['MINERU_DEVICE_MODE'] = 'cuda'

# Load models from local cache (downloaded in Cell 2.5). No network.
env['MINERU_MODEL_SOURCE'] = 'local'
env['HF_HUB_OFFLINE'] = '1'
env['TRANSFORMERS_OFFLINE'] = '1'



Safety belt: cap per-process VRAM estimate so MinerU stops over-allocating
and crashing on the 15GB T4. Chunking below is the real guarantee.


In [ ]:
env['MINERU_VIRTUAL_VRAM_SIZE'] = '8'



============ CHUNK SETTINGS ============
1-4 PDFs proven safe on a T4; 5+ OOMs. 3 leaves headroom for big papers.
If a chunk still fails -> set to 1. For a bit more speed -> try 4.


In [ ]:
CHUNK_SIZE = 3

def _chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

batch_input = Path('/content/mineru_batch_input')
temp_out = Path('/content/mineru_batch_output')
results = {}

n_chunks = (len(pdf_paths) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Converting {len(pdf_paths)} PDFs in {n_chunks} chunks of {CHUNK_SIZE}")
print(f"  Tables: ON | Formulas: OFF | Backend: pipeline")
print(f"  Model loads once per chunk (avoids the 5-PDF VRAM crash)\n")

start_total = time.time()

for ci, chunk in enumerate(_chunks(pdf_paths, CHUNK_SIZE), 1):
    # fresh input dir for this chunk
    if batch_input.exists():
        shutil.rmtree(batch_input)
    batch_input.mkdir()
    for orig_name, pdf_path in chunk:
        dst = batch_input / pdf_path.name
        try:
            os.symlink(pdf_path.absolute(), dst)
        except OSError:
            shutil.copy(pdf_path, dst)
    if temp_out.exists():
        shutil.rmtree(temp_out)

    print(f"[chunk {ci}/{n_chunks}] processing {len(chunk)} PDFs...", flush=True)
    t0 = time.time()
    try:
        proc = subprocess.run(
            [MINERU, '-p', str(batch_input), '-o', str(temp_out),
             '-b', 'pipeline', '-m', 'txt', '-l', 'ch'],
            capture_output=True, text=True, timeout=3600, env=env
        )
        if proc.returncode != 0:
            print(f"  ⚠ chunk {ci} exit code {proc.returncode}")
            print(f"  {proc.stderr[-600:]}")
    except subprocess.TimeoutExpired:
        print(f"  ✗ chunk {ci} timed out")

    # extract this chunk's outputs immediately
    for orig_name, pdf_path in chunk:
        orig_stem = Path(orig_name).stem
        paper_dir = temp_out / pdf_path.stem
        md_candidates = list(paper_dir.glob('**/*.md')) if paper_dir.exists() else []
        if md_candidates:
            src_md = md_candidates[0]
            shutil.copy(src_md, output_base / f"{orig_stem}.md")
            img_dirs = list(src_md.parent.glob('images'))
            if img_dirs and any(img_dirs[0].iterdir()):
                dst_img = output_base / f"{orig_stem}_images"
                if dst_img.exists():
                    shutil.rmtree(dst_img)
                shutil.copytree(img_dirs[0], dst_img)
                results[orig_name] = f"\u2713 {orig_stem}.md + {len(list(dst_img.iterdir()))} images"
            else:
                results[orig_name] = f"\u2713 {orig_stem}.md"
        else:
            results[orig_name] = "\u2717 failed (check chunk stderr above)"

    print(f"  chunk {ci} done in {(time.time()-t0)/60:.1f} min\n", flush=True)

# cleanup temp dirs
for p in [temp_out, batch_input]:
    if p.exists():
        shutil.rmtree(p)

elapsed = time.time() - start_total
ok = sum(1 for s in results.values() if s.startswith('\u2713'))
print(f"{'='*60}")
print(f"Total: {elapsed/60:.1f} min for {len(pdf_paths)} papers  ({ok} ok, {len(pdf_paths)-ok} failed)")
print(f"{'='*60}\n")
for name, status in results.items():
    print(f"  {status}")






***Gemni Image description generation part***


In [ ]:
"""***Gemni Image description generation part***"""
# @title
!pip install google-generativeai pillow -q

import google.generativeai as genai
from google.colab import userdata

try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except:
    API_KEY = input("Paste Gemini API key: ").strip()

genai.configure(api_key=API_KEY)
MODEL = genai.GenerativeModel('gemini-flash-lite-latest')

# Test
r = MODEL.generate_content("Say 'works'")
print(f"✓ {r.text}")

# @title
import re
import asyncio
from pathlib import Path
from PIL import Image
import time

output_base = Path('/content/markdown_output')

MIN_IMAGE_KB = 15
CONTEXT_CHARS = 500
MAX_PARALLEL = 1                # No point parallel with rate limit
RATE_LIMIT_DELAY = 4.5          # sec between calls (safe for 15 req/min)

PROMPT = """You are analyzing a figure from a scientific research paper.

Context from paper (text around figure):
{context}

Describe this figure precisely in 2-4 sentences. Focus on:
- What is shown (chart type, subject, key data)
- Key values, trends, findings
- Labels, axes, legends
- Scientific significance

Be concise. No preamble ("This figure shows..."). Start directly.
If logo/decoration/math snippet/too small, respond: SKIP"""

last_call_time = [0.0]
rate_lock = asyncio.Lock()


def get_context(md_text, pos, chars=CONTEXT_CHARS):
    start = max(0, pos - chars)
    end = min(len(md_text), pos + chars)
    ctx = md_text[start:end]
    return re.sub(r'!\[.*?\]\(.*?\)', '', ctx).strip()


async def describe_image(img_path, context, sem, max_retries=6):
    async with sem:
        # Rate limit — spread calls across time
        async with rate_lock:
            now = asyncio.get_event_loop().time()
            wait = RATE_LIMIT_DELAY - (now - last_call_time[0])
            if wait > 0:
                await asyncio.sleep(wait)
            last_call_time[0] = asyncio.get_event_loop().time()

        for attempt in range(max_retries):
            try:
                img = Image.open(img_path)
                prompt = PROMPT.format(context=context[:2000])
                response = await asyncio.to_thread(MODEL.generate_content, [prompt, img])
                desc = response.text.strip()
                return None if desc.upper().startswith('SKIP') else desc
            except Exception as e:
                if '429' in str(e) and attempt < max_retries - 1:
                    wait = 10 * (attempt + 1)
                    await asyncio.sleep(wait)
                    continue
                return None


async def enhance_markdown(md_path):
    md_text = md_path.read_text(encoding='utf-8')
    md_dir = md_path.parent

    img_pattern = re.compile(r'!\[([^\]]*)\]\(([^)]+)\)')
    matches = list(img_pattern.finditer(md_text))
    if not matches:
        return md_text, 0, 0, 0

    sem = asyncio.Semaphore(MAX_PARALLEL)
    tasks = []
    task_info = []
    skipped = 0

    for m in matches:
        img_rel_path = m.group(2)
        img_name = Path(img_rel_path).name
        candidates = [
            md_dir / img_rel_path,
            md_dir / f"{md_path.stem}_images" / img_name,
            md_dir / "images" / img_name,
        ]
        img_full_path = next((p for p in candidates if p.exists()), None)

        if img_full_path is None or img_full_path.stat().st_size / 1024 < MIN_IMAGE_KB:
            skipped += 1
            task_info.append((m, None))
            continue

        context = get_context(md_text, m.start())
        task = describe_image(img_full_path, context, sem)
        tasks.append(task)
        task_info.append((m, task))

    task_results = await asyncio.gather(*tasks) if tasks else []
    result_iter = iter(task_results)
    described = 0
    replacements = []

    for m, task in task_info:
        if task is not None:
            desc = next(result_iter)
            if desc:
                described += 1
                replacements.append((m.start(), m.end(), desc))

    new_md = md_text
    for start, end, text in sorted(replacements, key=lambda x: -x[0]):
        new_md = new_md[:start] + text + new_md[end:]

    return new_md, len(matches), described, skipped


async def process_all():
    md_files = sorted(output_base.glob('*.md'))
    md_files = [f for f in md_files if not f.name.endswith('_AI.md')]

    print(f"Enhancing {len(md_files)} files (rate-limited to 15 req/min)...\n")
    start = time.time()

    for i, md_path in enumerate(md_files, 1):
        t0 = time.time()
        print(f"[{i}/{len(md_files)}] {md_path.name}...")
        try:
            new_md, total, described, skipped = await enhance_markdown(md_path)
            out_path = md_path.parent / f"{md_path.stem}_AI.md"
            out_path.write_text(new_md, encoding='utf-8')
            print(f"    ✓ {described}/{total} replaced, {skipped} skipped in {time.time()-t0:.1f}s")
        except Exception as e:
            print(f"    ✗ {e}")

    print(f"\nTotal: {(time.time()-start)/60:.1f} min")


await process_all()



@title
=============================================
CELL 8: Download only AI-enhanced .md + images
=============================================


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

output_base = Path('/content/markdown_output')

# Build clean folder with only what we want
final_folder = Path('/content/final_output')
if final_folder.exists():
    shutil.rmtree(final_folder)
final_folder.mkdir()

# Copy AI-enhanced .md files, renamed WITHOUT the _AI suffix
ai_files = list(output_base.glob('*_AI.md'))
for ai_md in ai_files:
    # Strip _AI suffix so final file is just paper_name.md
    clean_name = ai_md.stem[:-3]  # remove "_AI"
    dst = final_folder / f"{clean_name}.md"
    shutil.copy(ai_md, dst)

# Copy image folders
for img_dir in output_base.glob('*_images'):
    dst = final_folder / img_dir.name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(img_dir, dst)

# ZIP the clean folder
shutil.make_archive('/content/papers_final', 'zip', final_folder)

# Report
md_count = sum(1 for _ in final_folder.glob('*.md'))
img_count = sum(1 for _ in final_folder.glob('*_images'))
print(f"✓ Final ZIP: {md_count} AI-enhanced markdown + {img_count} image folders")
print(f"  (Original .md files excluded)")

files.download('/content/papers_final.zip')

